# Perfil temporal de uma inferência com Prometheus

Este notebook envia uma requisição por vez para uma API OpenAI-compatible e correlaciona a resposta com a telemetria coletada no mesmo Pod. A região até o primeiro token é exibida como **TTFT/prefill observado**; o restante da geração é exibido como **decode observado**.

## 1. Antes de abrir o notebook

No Pod, use os alvos `observe-*` dentro de `services/inference-runtime`. Em um Pod novo, prepare o vLLM, Prometheus e notebook:

```bash
make observe-bootstrap OBS_RUNTIME=vllm
```

Depois mantenha três terminais no Pod, sempre com as mesmas variáveis:

```bash
make observe-serve OBS_RUNTIME=vllm
make observe-prometheus-start OBS_RUNTIME=vllm
make observe-jupyter OBS_RUNTIME=vllm OBS_JUPYTER_PORT=8889
```

O padrão para RTX 3090 é Qwen2.5-7B não quantizado, contexto 8192 e uma sequência. Para llama.cpp ou Ollama, troque `OBS_RUNTIME`; esses dois usam o GGUF configurado. O navegador pode chegar ao Jupyter por túnel SSH, mas o kernel e a requisição medida permanecem no Pod.

In [1]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in candidates if (path / 'services/inference-runtime/prometheus_profile.py').exists())
RUNTIME_DIR = REPO_ROOT / 'services/inference-runtime'
sys.path.insert(0, str(RUNTIME_DIR))

from prometheus_profile import (
    default_api_key, list_models, plot_profile, profile_prompts,
    profile_request, prometheus_ready, save_profile, summary_frame, telemetry_frame,
)


## 2. Configure o runtime

Exemplos usuais: vLLM em `8000`, llama-server em `8080` e Ollama em `11434`. O modelo pode ser quantizado ou não; basta estar servido pela API.

In [2]:
BASE_URL = os.environ.get('INFERENCE_BASE_URL', 'http://127.0.0.1:8000')
PROMETHEUS_URL = os.environ.get('PROMETHEUS_URL', 'http://127.0.0.1:9090')
MODEL = os.environ.get('INFERENCE_MODEL', '')  # vazio seleciona o primeiro /v1/models
API_KEY = default_api_key()
MAX_TOKENS = 256
PROMPT_BASE = (
    'Analise como CPU, GPU, RAM, VRAM e KV cache influenciam prefill e decode. '
    'Relacione cada recurso com TTFT e tokens por segundo, usando exemplos. '
)
PROMPT = PROMPT_BASE * 40  # contexto maior deixa a fase de prefill mais observável


## 3. Verifique as duas APIs

In [3]:
assert prometheus_ready(PROMETHEUS_URL), f'Prometheus indisponível em {PROMETHEUS_URL}'
models = list_models(BASE_URL, API_KEY)
assert models, f'Nenhum modelo encontrado em {BASE_URL}/v1/models'
if not MODEL:
    MODEL = models[0]
{'prometheus': 'ready', 'models': models, 'selected_model': MODEL}


AssertionError: Prometheus indisponível em http://127.0.0.1:9090

## 4. Execute uma requisição perfilada

A função espera alguns segundos após a resposta para que o último scrape chegue ao Prometheus. Esse tempo não entra na latência da requisição.

In [ ]:
run = profile_request(
    base_url=BASE_URL,
    model=MODEL,
    messages=[{'role': 'user', 'content': PROMPT}],
    prometheus_url=PROMETHEUS_URL,
    max_tokens=MAX_TOKENS,
    api_key=API_KEY,
)
summary_frame(run)


In [ ]:
print(run['request']['output'])

## 5. Analise prefill/TTFT e decode nos mesmos gráficos

In [ ]:
figure = plot_profile(run)
figure

## 6. Preserve o resultado bruto

Cada execução salva `profile.json`, `profile.png`, `request-summary.csv` e `telemetry.csv` em `results/notebook-prometheus/<timestamp>/`.

In [ ]:
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = RUNTIME_DIR / 'results' / 'notebook-prometheus' / stamp
run_dir.mkdir(parents=True, exist_ok=True)
save_profile(run, run_dir / 'profile.json')
summary_frame(run).to_csv(run_dir / 'request-summary.csv', index=False)
telemetry_frame(run).to_csv(run_dir / 'telemetry.csv', index=False)
figure.savefig(run_dir / 'profile.png', dpi=160, bbox_inches='tight')
run_dir


## 7. Opcional: repita prompts sequencialmente

Não há concorrência: a próxima requisição começa somente depois que a anterior e sua telemetria terminam.

In [ ]:
prompts = [
    'Explique em detalhes a diferença entre RAM e VRAM.',
    'Explique como o comprimento do contexto afeta o KV cache.',
]
runs = profile_prompts(
    prompts, base_url=BASE_URL, model=MODEL, prometheus_url=PROMETHEUS_URL,
    max_tokens=MAX_TOKENS, api_key=API_KEY,
)
summary_frame(runs)


## 8. Ao terminar

Encerre apenas a pilha de observação iniciada pelo projeto:

```bash
cd services/inference-runtime
make prometheus-stop
```